# Lay 1 cap tien te

In [ ]:
import sys
sys.path.append('../../Common/') # .. la directory cha của Common

import CommonMT5DWH

In [ ]:
import MetaTrader5 as mt5

from datetime import datetime, timedelta

symbol = 'EURUSD.sml'
from_date = datetime.strptime('1990-01-01', '%Y-%m-%d')
to_date = datetime.strptime('2025-09-24', '%Y-%m-%d')

timeframe = mt5.TIMEFRAME_D1  # 1 ngày

data = CommonMT5DWH.CommonMT5DWH.loaddataMT5_FromTo_List_Ext(symbol, from_date, to_date, timeframe)

data

In [3]:
data

,Symbol,Datetime,Open,High,Low,Close,Volume
0,EURUSD.sml,1990-01-02,1.16240,1.16360,1.14020,1.14260,2701
1,EURUSD.sml,1990-01-03,1.14230,1.14310,1.13090,1.13090,1301
2,EURUSD.sml,1990-01-04,1.13010,1.16950,1.12770,1.16720,4651
3,EURUSD.sml,1990-01-05,1.16730,1.16940,1.15120,1.15920,2831
4,EURUSD.sml,1990-01-08,1.16250,1.17460,1.15460,1.17240,3011
...,...,...,...,...,...,...,...
9276,EURUSD.sml,2025-09-17,1.18606,1.19189,1.18079,1.18135,163576
9277,EURUSD.sml,2025-09-18,1.18125,1.18484,1.17502,1.17862,175339
9278,EURUSD.sml,2025-09-19,1.17873,1.17928,1.17289,1.17452,155813
9279,EURUSD.sml,2025-09-22,1.17397,1.18033,1.17263,1.18024,121606


In [ ]:
# Kiem tra du lieu
# Xuat ra csv de check du lieu
data.to_csv('data.csv', index=False)

# Dung dataframe de check trung du lieu Symbol	Datetime	Open	High	Low	Close	Volume
duplicates = data[data.duplicated()]

if len(duplicates) > 0:
    print("\nCác dòng trùng lặp:")
    print(duplicates)
else:
    print("Không có dòng trùng lặp!")

# Nhóm dữ liệu theo bội số của 5, bỏ các cây đầu tiên dư ra
print(f"\nTổng số dòng: {len(data)}")

# Tính số dòng dư ra khi chia cho 5
remainder = len(data) % 5
print(f"Số dòng dư ra: {remainder}")

# Bỏ các dòng đầu tiên dư ra
if remainder > 0:
    data_grouped = data.iloc[remainder:].copy()
    print(f"Sau khi bỏ {remainder} dòng đầu: {len(data_grouped)} dòng")
else:
    data_grouped = data.copy()
    print("Không cần bỏ dòng nào")

# Thêm cột group_id để nhóm theo 5
data_grouped['group_id'] = (data_grouped.index - remainder) // 5

print(f"\nSố nhóm được tạo: {data_grouped['group_id'].nunique()}")
print(f"Dữ liệu sau khi nhóm:")
print(data_grouped.head(10))

# Xuất dữ liệu đã nhóm ra CSV
data_grouped.to_csv('data_grouped_by_5.csv', index=False)
print("\nĐã xuất dữ liệu đã nhóm ra file: data_grouped_by_5.csv")

In [ ]:
# Hiển thị chi tiết các nhóm và thống kê
print("=== THỐNG KÊ THEO NHÓM ===")

# Thống kê số lượng dòng trong mỗi nhóm
group_counts = data_grouped['group_id'].value_counts().sort_index()
print(f"\nSố dòng trong mỗi nhóm:")
print(group_counts.head(10))

# Kiểm tra xem tất cả nhóm đều có đúng 5 dòng không
print(f"\nTất cả nhóm đều có 5 dòng: {all(group_counts == 5)}")

# Hiển thị một vài nhóm mẫu
print(f"\n=== NHÓM MẪU ===")
for group_id in [0, 1, 2]:
    group_data = data_grouped[data_grouped['group_id'] == group_id]
    print(f"\nNhóm {group_id} ({len(group_data)} dòng):")
    print(group_data[['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']].to_string(index=False))

# Tính toán thống kê cho từng nhóm (ví dụ: giá trung bình, volume trung bình)
print(f"\n=== THỐNG KÊ THEO NHÓM ===")
group_stats = data_grouped.groupby('group_id').agg({
    'Open': 'mean',
    'High': 'max', 
    'Low': 'min',
    'Close': 'mean',
    'Volume': 'sum'
}).round(5)

print("Thống kê 10 nhóm đầu tiên:")
print(group_stats.head(10))
